NECESSARY SET UP

In [ ]:
!pip install spektral
import numpy as np
import networkx as nx
import pandas as pd
import tensorflow as tf
import pickle

from spektral.layers import GCNConv 
from spektral.layers import GATConv

from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

from tensorflow.keras import Model 
from tensorflow.keras.layers import Dense, Dropout, Input

from tensorflow.keras.models import Sequential

from sklearn.cluster import KMeans
from tensorflow.python.client import device_lib

tsne_model = TSNE(learning_rate=100)


In [ ]:
print(f"TensorFlow Version: {tf.__version__}")

physical_devices = tf.config.list_physical_devices('GPU')
print("Num GPUs Available: ", len(physical_devices))

# This ensures TensorFlow uses it efficiently
if len(physical_devices) > 0:
    try:
        tf.config.experimental.set_memory_growth(physical_devices[0], True)
        print("GPU Memory Growth enabled")
    except:
        pass

THE EMBEDDER MODEL

THE GCN ENCODER (MESSAGE PASSING ARCHITECHTURE)

In [ ]:
def build_encoder(embedding_size, num_features, num_hyperedges):
    """
    GNN Encoder: Handles any number of node features and group structures.
    """
    # 1. Generic Inputs
    x_in = Input(shape=(num_features,), name="Node_Features") # Generic node data 
    a_in = Input(shape=(None,), sparse=True, name="Pairwise_Adj") # Generic graph structure 
    h_in = Input(shape=(num_hyperedges,), name="Hypergraph_Incidence") # Generic group membership 

    #  Pairwise Graph Attention (GAT)
    # Multi-head attention (4 heads) to capture varied node relationships [cite: 24]
    x_gat = GATConv(embedding_size * 2, attn_heads=4, concat=True, activation='relu')([x_in, a_in])
    
    # Hypergraph Higher-Order Flow 
    # Captures influence pathways within shared environments (Groups/Hyperedges) 
    # Logic: Information travels from Node -> Group -> Node regardless of group type [cite: 32]
    # Mathematical flow: H * (H_T * X)
    group_agg = tf.matmul(h_in, tf.matmul(h_in, x_in, transpose_a=True))
    x_hyper = Dense(embedding_size * 2 * 4, activation='relu')(group_agg)

    # Combining Pairwise and Group-level signals 
    # This ensures the model captures both "bridge" edges and community clusters 
    x_combined = tf.add(x_gat, x_hyper)

    # 3. Final Output Layer: Produces the 'Fair' and 'Group-Aware' embedding
    z = GATConv(embedding_size, attn_heads=4, concat=False, activation='relu')([x_combined, a_in])

    return Model(inputs=[x_in, a_in, h_in], outputs=z)

THE GCN DECODER

In [ ]:
def build_decoder(embedding_size, output_size):

    model = Sequential()

    # Expand the embedding back toward the original graph dimensions
    model.add(Dense(embedding_size * 2, activation='relu'))
    model.add(Dense(embedding_size * 4, activation='relu'))

    # Output a probability for every possible node connection
    model.add(Dense(output_size, activation='sigmoid'))

    return model

TOPLOGY AWARE GNN AUTO ENCODER ASSEMBLY

In [ ]:
def build_ae(encoder, decoder, n, num_features, num_hyperedges):
    # Define all three required GNN inputs
    features_in = Input(shape=(num_features,), name="Node_Features")
    adj_in = Input(shape=(n,), name="Graph_Structure", sparse=True)
    h_in = Input(shape=(num_hyperedges,), name="Hypergraph_Incidence")

    # Pass all three to the encoder (Message Passing + Higher-Order Flow)
    embeddings = encoder([features_in, adj_in, h_in])
    
    # Reconstruct the structure
    reconstructions = decoder(embeddings)

    auto_encoder = Model(inputs=[features_in, adj_in, h_in], outputs=reconstructions)

    return auto_encoder

GNN AUTO ENCODER LOSS & ACCURACY

In [ ]:
def recon_loss(x, x_hat):
    # This loss measures how well the GNN captures the edges from edges.txt
    return tf.reduce_sum(tf.keras.losses.binary_crossentropy(x, x_hat))

In [ ]:
def first_order_loss(X, Z):
    # X: The Adjacency Matrix (Graph Structure)
    # Z: The Embeddings generated by the GCN Encoder
    X = tf.cast(X, tf.float32)
    Z = tf.cast(Z, tf.float32)

    # Compute the Degree Matrix and the Graph Laplacian
    D = tf.linalg.diag(tf.reduce_sum(X, 1))
    L = D - X  # L captures the 'flow' or connectivity of the graph

    # Trace calculation ensures that connected nodes maintain similar representations
    return 2 * tf.linalg.trace(tf.matmul(tf.matmul(tf.transpose(Z), L), Z))

In [ ]:
def covariance_loss(embeddings, sensitive_labels):
    
    # Cast to float32 for matrix operations
    Z = tf.cast(embeddings, tf.float32)
    # Ensure sensitive_labels is 2D (batch_size, 1)
    S = tf.reshape(tf.cast(sensitive_labels, tf.float32), [-1, 1])
    
    # 1. Mean Center the data
    z_centered = Z - tf.reduce_mean(Z, axis=0)
    s_centered = S - tf.reduce_mean(S, axis=0)
    
    # 2. Matrix Multiplication to get Covariance Matrix
    # Cov = (Z^T * S) / (n - 1)
    n = tf.cast(tf.shape(Z)[0], tf.float32)
    cov_matrix = tf.matmul(tf.transpose(z_centered), s_centered) / (n - 1.0)
    
    # 3. Frobenius Norm: Square root of the sum of squares of all elements
    return tf.linalg.norm(cov_matrix, ord='fro')

In [ ]:
def drop_edges(adj_matrix, drop_rate=0.15):
    """
    Randomly removes edges from the adjacency matrix to create an augmented view. 
    """

    mask = tf.random.uniform(tf.shape(adj_matrix)) > drop_rate

    return tf.cast(adj_matrix, tf.float32) * tf.cast(mask, tf.float32)

In [ ]:
def contrastive_loss(z_original, z_augmented, temperature=0.5):
    """
    Computes the NT-Xent loss to ensure consistency across augmented views. 
    """
    # Normalize embeddings to the unit hypersphere
    z_original = tf.math.l2_normalize(z_original, axis=1)
    z_augmented = tf.math.l2_normalize(z_augmented, axis=1)
    
    # Calculate cosine similarity between all pairs
    # Positive pairs are the diagonal elements (same node, different view)
    sim_matrix = tf.matmul(z_original, z_augmented, transpose_b=True) / temperature
    
    # We want to maximize the diagonal values relative to the off-diagonal values
    labels = tf.range(tf.shape(z_original)[0])
    loss = tf.keras.losses.sparse_categorical_crossentropy(labels, sim_matrix, from_logits=True)
    
    return tf.reduce_mean(loss)

In [ ]:
def ae_adversarial_loss(reconstructions, originals, discriminators_outputs, embeddings, sensitive_labels_list, z_aug_list=None):
    """
    Generic Loss: Handles N sensitive attributes and optional Contrastive Learning.
    """
    # 1. Base Reconstruction Loss 
    total_loss = recon_loss(originals, reconstructions)
    
    # 2. Dynamic Fairness & Robustness Loop
    for i, (d_out, s_label) in enumerate(zip(discriminators_outputs, sensitive_labels_list)):
        
        # Adversarial Loss
        # Discriminator tries to guess, Encoder tries to scrub.
        total_loss += 2 * tf.reduce_sum(tf.keras.losses.binary_crossentropy(tf.ones_like(d_out), d_out))
        
        # Covariance Penalty - Mathematically enforces linear independence.
        total_loss += 10 * covariance_loss(embeddings, s_label)
        
        # Graph Contrastive Loss 
        if z_aug_list is not None:
            total_loss += 5 * contrastive_loss(embeddings, z_aug_list[i])
        
    return total_loss

In [ ]:
# GNN Topology Reconstruction Accuracy
def ae_accuracy(x, x_hat):
    # Round predicted edge probabilities to binary values (0 or 1)
    round_x_hat = tf.round(x_hat)
    
    # Compare GNN-reconstructed edges against the original graph structure
    return tf.reduce_mean(tf.cast(tf.equal(x, round_x_hat), tf.float32))

PRE TRAINING STEPS FOR GRAPH AWARE EMBEDDINGS

In [ ]:
# GNN-Based Pre-training Step for Graph-Aware Embeddings
def pretrain_step_embd(adj_matrix, batch_features, full_features, encoder, decoder, auto_encoder, pre_optimizer, first_order, alpha):
   
   
        # Encoder uses Message Passing to create embeddings based on neighborhood structure
        z = encoder([batch_features, adj_matrix], training=True)
        x_hat = decoder(z, training=True)

        # Generate embeddings for the entire graph to calculate global smoothness
        Z = encoder([full_features, adj_matrix], training=True)

        # Standard reconstruction loss for the edges in the batch
        pre_loss = recon_loss(batch_features, x_hat)

        # Incorporate the Graph Laplacian to preserve proximity between neighbors
        if first_order == 'with_f1':
            pre_loss += alpha * first_order_loss(adj_matrix, Z)

    # Update the GCN layers and MLP decoder layers simultaneously
    pre_gradients = pre_tape.gradient(pre_loss, auto_encoder.trainable_variables)
    pre_optimizer.apply_gradients(zip(pre_gradients, auto_encoder.trainable_variables))

    # Calculate accuracy of the graph reconstruction
    pre_acc = ae_accuracy(batch_features, x_hat)

    return tf.reduce_mean(pre_loss), pre_acc

In [ ]:
# GNN-Based Pre-training Loop with Neighborhood Awareness

def pretrain_embd(adj_matrix, feature_matrix, idxs, encoder, decoder, auto_encoder, pre_optimizer, first_order, alpha):
    np.random.shuffle(idxs)
    PRE_EPOCHS = 300
    Batch_size = 50

    for epoch in range(PRE_EPOCHS):
        epoch_losses = []
        epoch_acc = []

        for batch_idx in range(0, len(idxs), Batch_size):
            # Select the indices for the current batch
            selected_idxs = idxs[batch_idx: batch_idx + Batch_size]
            
            # Extract the features for these specific nodes
            batch_features = feature_matrix[selected_idxs, :]

            # Execute the pre-training step
            loss, accuracy = pretrain_step_embd(adj_matrix, 
                                                tf.cast(batch_features, tf.float32), 
                                                tf.cast(feature_matrix, tf.float32), 
                                                encoder, decoder, auto_encoder,
                                                pre_optimizer, first_order, alpha)

            epoch_losses.append(loss)
            epoch_acc.append(accuracy)

        if epoch % 50 == 0:
            print(f"Epoch {epoch}: Loss is {np.array(epoch_losses).mean():.4f}, Accuracy: {np.array(epoch_acc).mean():.4f}")

In [ ]:
@tf.custom_gradient
def gradient_reversal(x, alpha=1.0):
    """
    Forward Pass: Identity (does nothing). 
    Backward Pass: Multiplies gradient by -alpha. 
    """
    def grad(dy):
        return -dy * alpha, None
    return x, grad

class GradientReversalLayer(tf.keras.layers.Layer):
    def __init__(self, alpha=1.0, **kwargs):
        super().__init__(**kwargs)
        self.alpha = alpha

    def call(self, x):
        return gradient_reversal(x, self.alpha)

ADVERSARIAL MLP DISCRIMINATOR FOR GNN EMBEDDING EVALUATION

In [ ]:
# Adversarial MLP Discriminator with Gradient Reversal (GRL)
def build_discriminator(embedding_size, alpha=1.0):
    # The Discriminator evaluates the 'fairness' of the embeddings (Z)
    model = Sequential()

    # Input: The latent embedding Z produced by the GCN Message Passing
    model.add(Input(shape=(embedding_size,)))


    # This layer flips the "error signal" between the Encoder and Discriminator.
    # It forces the Encoder to actively scrub sensitive information away.
    model.add(GradientReversalLayer(alpha=alpha))

    # Hidden Layers: Extracting demographic patterns from the GNN output
    model.add(Dense(25, activation='relu'))
    model.add(Dropout(0.25))

    model.add(Dense(15, activation='relu'))
    model.add(Dropout(0.20))

    model.add(Dense(6, activation='relu'))
    model.add(Dropout(0.20))

    # Output: Probability of a sensitive attribute (e.g., Age < 20)
    model.add(Dense(1, activation='sigmoid'))

    return model

In [ ]:
# Adversarial Group-Classification Loss for Fair GNN Embeddings
def disc_loss_function(d_z0, d_z1):
    # This loss measures how accurately the Discriminator can 
    # extract sensitive attributes from GNN-generated embeddings
    
    # Calculate loss for the first demographic group (Label 0)
    loss_zero = tf.keras.losses.binary_crossentropy(tf.zeros_like(d_z0), d_z0)
    
    # Calculate loss for the second demographic group (Label 1)
    loss_one = tf.keras.losses.binary_crossentropy(tf.ones_like(d_z1), d_z1)

    # Returning the sum forces the discriminator to be equally good at 
    # identifying both groups to ensure high 'fairness' pressure on the GNN
    return tf.cast(loss_zero, tf.float32) + tf.cast(loss_one, tf.float32)

JOINT ADVERSARIAL TRAINING STEP FOR FAIR GNN EMBEDDINGS

In [ ]:
def train_step(adj_matrix, h_matrix, x10, x11, x20, x21, encoder, decoder, auto_encoder, discriminator1, discriminator2, ae_optimizer):
    
    # 1. Generate the augmented (distorted) view for Graph Contrastive Learning (GCL) 
    adj_augmented = drop_edges(adj_matrix, drop_rate=0.15)
    
    # Using GradientTape to track all operations for backpropagation
    with tf.GradientTape() as tape:
        # 2. Forward Pass (Original Graph)
        z10 = encoder([x10, adj_matrix, h_matrix], training=True)
        z11 = encoder([x11, adj_matrix, h_matrix], training=True)
        z20 = encoder([x20, adj_matrix, h_matrix], training=True)
        z21 = encoder([x21, adj_matrix, h_matrix], training=True)

        # 3. Forward Pass (Augmented Graph) for Consistency/Contrastive Learning
        z10_aug = encoder([x10, adj_augmented, h_matrix], training=True)
        z11_aug = encoder([x11, adj_augmented, h_matrix], training=True)
        z20_aug = encoder([x20, adj_augmented, h_matrix], training=True)
        z21_aug = encoder([x21, adj_augmented, h_matrix], training=True)

        # 4. Discriminators with GRL (Adversarial Fairness)
        d_z10 = discriminator1(z10, training=True)
        d_z11 = discriminator1(z11, training=True)
        d_z20 = discriminator2(z20, training=True)
        d_z21 = discriminator2(z21, training=True)

        # 5. Decoder Reconstruction
        x10_hat = decoder(z10, training=True)
        x11_hat = decoder(z11, training=True)   
        x20_hat = decoder(z20, training=True)
        x21_hat = decoder(z21, training=True)

        # 6. Prepare Multi-Attribute Sensitive Labels
        s1 = tf.concat([tf.zeros(tf.shape(z10)[0]), tf.ones(tf.shape(z11)[0])], axis=0)
        s2 = tf.concat([tf.zeros(tf.shape(z20)[0]), tf.ones(tf.shape(z21)[0])], axis=0)

        # 7. Unified Loss Preparation
        all_reconstructions = tf.concat([x10_hat, x11_hat, x20_hat, x21_hat], 0)
        all_originals = tf.concat([x10, x11, x20, x21], 0)
        all_embeddings = tf.concat([z10, z11, z20, z21], 0)

        # 8. Calculating Unified Adversarial Loss
        ae_loss = ae_adversarial_loss(
            all_reconstructions, 
            all_originals, 
            [tf.concat([d_z10, d_z11], 0), tf.concat([d_z20, d_z21], 0)], 
            all_embeddings, 
            [s1, s2],
            [tf.concat([z10_aug, z11_aug], 0), tf.concat([z20_aug, z21_aug], 0)]
        )

    # 9. Minimax Optimization: Calculate and Apply Gradients
    all_trainable_vars = (auto_encoder.trainable_variables + 
                          discriminator1.trainable_variables + 
                          discriminator2.trainable_variables)
    
    gradients = tape.gradient(ae_loss, all_trainable_vars)
    ae_optimizer.apply_gradients(zip(gradients, all_trainable_vars))

    # 10. Metric tracking: Measure how well the GNN reconstructs the physical links
    ae_acc = ae_accuracy(all_originals, all_reconstructions)

    return tf.reduce_mean(ae_loss), ae_acc

PRE TRAINING THE DISCRIMINATOR

In [ ]:
# GNN-Aware Discriminator Pre-training Step
def pretrain_step_disc(adj_matrix, x0, x1, encoder, discriminator, disc_pre_optimizer):

    z0 = encoder([x0, adj_matrix])
    z1 = encoder([x1, adj_matrix])

    with tf.GradientTape() as disc_tape_sep:
        # The discriminator evaluates the fairness of the GNN's neighbor aggregation
        d_z0 = discriminator(z0, training=True)
        d_z1 = discriminator(z1, training=True)

        # Calculate how well the discriminator identifies sensitive traits
        disc_loss = disc_loss_function(d_z0, d_z1)

    # Compute and apply gradients to update the Discriminator's ability to spot bias
    gradients_disc = disc_tape_sep.gradient(disc_loss, discriminator.trainable_variables)
    disc_pre_optimizer.apply_gradients(zip(gradients_disc, discriminator.trainable_variables))

    return tf.reduce_mean(disc_loss)

In [ ]:
# GNN-Aware Discriminator Pre-training Loop
def pretrain_disc(adj_matrix, feature_matrix, idxs_zeros, idxs_ones, encoder, discriminator, disc_pre_optimizer):
    EPOCHS = 40

    # Shuffle both demographic groups to ensure randomized, balanced training
    np.random.shuffle(idxs_zeros)
    np.random.shuffle(idxs_ones)
    Batch_size = 50

    for epoch in range(EPOCHS):
        # Iterate through the balanced indices
        for batch_idx in range(0, len(idxs_ones), Batch_size):
            selected_zeros = idxs_zeros[batch_idx: batch_idx + Batch_size]
            selected_ones = idxs_ones[batch_idx: batch_idx + Batch_size]

            # Extract the features for the current batch of nodes
            x0 = feature_matrix[selected_zeros]
            x1 = feature_matrix[selected_ones]

            pretrain_step_disc(adj_matrix, x0, x1, encoder, discriminator, disc_pre_optimizer)

THE TRAINING LOOP

In [ ]:
# Topology-Aware Adversarial Training Loop for Fair GNNs
def adversarial_train(adj_matrix, feature_matrix, idxs1_zeros, idxs1_ones, idxs2_zeros, idxs2_ones, encoder, decoder, auto_encoder, discriminator1, 
                      discriminator2, ae_optimizer, disc1_optimizer, disc2_optimizer):
    EPOCHS = 500

    # Shuffle all group indices to ensure randomized training across epochs
    np.random.shuffle(idxs1_zeros)
    np.random.shuffle(idxs1_ones)
    np.random.shuffle(idxs2_zeros)
    np.random.shuffle(idxs2_ones)

    Batch_size = 50

    for epoch in range(EPOCHS):
        # Iterate through the groups using the length of the target attribute list
        for batch_idx in range(0, len(idxs1_ones), Batch_size):
            selected_zeros1 = idxs1_zeros[batch_idx: batch_idx + Batch_size]
            selected_ones1 = idxs1_ones[batch_idx: batch_idx + Batch_size]
            selected_zeros2 = idxs2_zeros[batch_idx: batch_idx + Batch_size]
            selected_ones2 = idxs2_ones[batch_idx: batch_idx + Batch_size]

            # Extract node features instead of raw adjacency rows
            x10 = feature_matrix[selected_zeros1]
            x11 = feature_matrix[selected_ones1]
            x20 = feature_matrix[selected_zeros2]
            x21 = feature_matrix[selected_ones2]

            # Joint Training: Pass the full adj_matrix to allow GCN Message Passing
            train_step(adj_matrix, 
                       tf.cast(x10, tf.float32), tf.cast(x11, tf.float32), 
                       tf.cast(x20, tf.float32), tf.cast(x21, tf.float32),
                       encoder, decoder, auto_encoder, discriminator1, discriminator2, 
                       ae_optimizer, disc1_optimizer, disc2_optimizer)

SEED SELECTION

In [ ]:
# Topology-Aware Seed Selection for Fair Influence Maximization
def get_seeds(N_CLUS, embedding, nodes, labels, nodes_zero, nodes_one, strategy, n_seeds):

    # Cluster the GNN-generated fair embeddings
    model = KMeans(n_clusters=N_CLUS)
    model.fit(embedding)

    cluster_number = model.labels_
    centers = model.cluster_centers_

    seed_ids = [[] for i in range(N_CLUS)]

    for i in range(N_CLUS):
        # Strategy: Select nodes closest to GNN cluster centroids
        if strategy == 'nearest':
            # Distance is calculated in the fair latent space created by the GCN
            sorted_distance = np.array(sorted(
                [[np.sqrt(np.sum(np.power(centers[i] - embedding[j], 2))), j] for j in range(len(embedding)) if
                 i == cluster_number[j]]))
            seed_ids[i].extend(list(sorted_distance[:n_seeds, 1]))

        # Strategy: Maintain demographic balance within topological clusters
        elif strategy == 're-cluster':
            temp = []
            sorted_distance = np.array(sorted(
                [[np.sqrt(np.sum(np.power(centers[i] - embedding[j], 2))), j] for j in range(len(embedding)) if
                 i == cluster_number[j]]))
            temp.extend(list(sorted_distance[:n_seeds, 1]))

            portion_zero = sum(1 for num in temp if num in nodes_zero)
            portion_one = sum(1 for num in temp if num in nodes_one)

            # Subset the fair GNN embeddings by cluster and demographic label
            zero_in_clus = embedding[np.logical_and(cluster_number == i, labels == 0)]
            zero_inds = nodes[np.logical_and(cluster_number == i, labels == 0)]

            one_in_clus = embedding[np.logical_and(cluster_number == i, labels == 1)]
            one_inds = nodes[np.logical_and(cluster_number == i, labels == 1)]

            # Re-center selection within the GNN fair latent space for group 0
            if len(zero_in_clus) != 0:
                model_on_zero = KMeans(n_clusters=1)
                model_on_zero.fit(zero_in_clus)
                center_zero = model_on_zero.cluster_centers_

                sorted_distance_zero = np.array(sorted(
                    [[np.sqrt(np.sum(np.power(center_zero - zero_in_clus[j], 2))), j] for j in
                     range(len(zero_in_clus))]))
                seed_ids[i].extend([zero_inds[int(idx)] for idx in sorted_distance_zero[:portion_zero, 1]])

            # Re-center selection within the GNN fair latent space for group 1
            if len(one_in_clus) != 0:
                model_on_one = KMeans(n_clusters=1)
                model_on_one.fit(one_in_clus)
                center_one = model_on_one.cluster_centers_

                sorted_distance_one = np.array(sorted(
                    [[np.sqrt(np.sum(np.power(center_one - one_in_clus[j], 2))), j] for j in range(len(one_in_clus))]))
                seed_ids[i].extend([one_inds[int(idx)] for idx in sorted_distance_one[:portion_one, 1]])

    return np.reshape(seed_ids, newshape=(-1,))

THE IC ALGORITHM

In [ ]:
# Simulation of Information Spread via Independent Cascade (IC)
def IC(G, seeds, imp_prob, recover_prob=0, remove=0):

    impressed = []
    removed = []
    front = list(seeds[:])

    while front:
        impressed.extend(front)
        impressed = np.array(impressed)

        # Handle recovery/removal dynamics 
        if recover_prob != 0:
            random_draws = np.random.uniform(size=len(impressed))
            if remove:
                removed.extend(impressed[random_draws < recover_prob])
                removed = list(set(removed))
            impressed = impressed[random_draws >= recover_prob]

        impressed = list(impressed)
        new_front = []

        # Neighborhood traversal for influence spread
        for node in front:
            # GNN-aware spreading relies on the physical links in the graph G
            neighbours = list(G.neighbors(node))

            for neigh in neighbours:
                # Stochastic trial for influence success
                expr_prob = np.random.uniform(size=1)[0]
                
                # Ensure node is only influenced once and isn't removed
                if expr_prob < imp_prob and not (neigh in impressed) and not (neigh in new_front) and not (
                        neigh in removed):
                    new_front.append(neigh)

        # Move to the next "hop" in the network
        front = new_front[:]

    impressed = np.reshape(np.array(impressed), newshape=(-1,))

    return impressed

REPEATED IC

In [ ]:
# Statistical Evaluation of Multi-Attribute Fair Influence Spread
def repeated_IC(G, seeds, seed_type, n_expr, imp_prob):
   
    zeros_count1, ones_count1 = [], []
    zeros_count2, ones_count2 = [], []
    total_count = []

    for i in range(n_expr):
        # Running the IC model using seeds generated from GNN clustering
        impressed = IC(G, seeds, imp_prob)
        total_count.append(len(impressed))

        count_zeros1, count_ones1 = 0, 0
        count_zeros2, count_ones2 = 0, 0

        # Auditing the demographic reach of the influence spread
        for imp in impressed:
            if imp in attr1_zero: count_zeros1 += 1
            elif imp in attr1_one: count_ones1 += 1
                
            if imp in attr2_zero: count_zeros2 += 1
            elif imp in attr2_one: count_ones2 += 1

        zeros_count1.append(count_zeros1)
        ones_count1.append(count_ones1)
        zeros_count2.append(count_zeros2)
        ones_count2.append(count_ones2)

    # Calculating average reach and normalized influence fractions
    total_imp = np.round(np.mean(total_count), 2)
    total_fraction = np.round(total_imp / len(G.nodes()), 3)

    # Fractions for Attribute 1 
    fraction_zero1 = np.round(np.mean(zeros_count1) / len(attr1_zero), 3)
    fraction_one1 = np.round(np.mean(ones_count1) / len(attr1_one), 3)
    
    # Fractions for Attribute 2 
    fraction_zero2 = np.round(np.mean(zeros_count2) / len(attr2_zero), 3)
    fraction_one2 = np.round(np.mean(ones_count2) / len(attr2_one), 3)

    return total_imp, total_fraction, fraction_zero1, fraction_one1, fraction_zero2, fraction_one2

In [ ]:
# GNN-Based Raw Influence Data Extraction for Multi-Attribute Evaluation
def get_IC_influenced(G, seeds, n_expr, imp_prob):

    zeros_count1, ones_count1 = [], []
    zeros_count2, ones_count2 = [], []
    total_count = []

    for i in range(n_expr):
        # Simulation using topology-aware seeds from the GCN model
        impressed = IC(G, seeds, imp_prob)
        total_count.append(len(impressed))

        c0_attr1, c1_attr1 = 0, 0
        c0_attr2, c1_attr2 = 0, 0

        # Categorizing influenced nodes by GNN-protected attributes
        for imp in impressed:
            # Check Attribute 1
            if imp in attr1_zero: c0_attr1 += 1
            elif imp in attr1_one: c1_attr1 += 1
            
            # Check Attribute 2 
            if imp in attr2_zero: c0_attr2 += 1
            elif imp in attr2_one: c1_attr2 += 1

        zeros_count1.append(c0_attr1)
        ones_count1.append(c1_attr1)
        zeros_count2.append(c0_attr2)
        ones_count2.append(c1_attr2)

    # Returning structured arrays for all experiments to allow further statistical analysis
    return (np.array(total_count), 
            np.array(zeros_count1), np.array(ones_count1), 
            np.array(zeros_count2), np.array(ones_count2))

GRAPH & FEATURE MATRIX CONSTRUCTION

In [ ]:
# GNN-Ready Graph and Feature Matrix Construction
def get_graph_real():
    # Load the social network topology from edges.txt 
    graph_df = pd.read_csv('edges.txt', sep="\t", header=None)
    graph_df.columns = ['s', 't']

    # Load node attributes 
    attr_df = pd.read_csv('attr.txt', sep="\t", header=None)
    
    # input_G represents the raw global network
    edges = [(row.s, row.t) for _, row in graph_df.iterrows()]
    input_G = nx.from_edgelist(edges)

    extra_nodes = attr_df[attr_df.iloc[:, 2] > 20].iloc[:, 0].tolist()
    
    unfrozen_G = nx.Graph(input_G)
    for node in input_G.nodes():
        if node in extra_nodes:
            # Removing nodes that don't fit the research criteria 
            unfrozen_G.remove_node(node) 
    
    # Generate the Adjacency Matrix (A) for GAT/GCN Message Passing 
    A = nx.to_numpy_array(unfrozen_G)
    G = nx.from_numpy_array(A)

    return G, A, unfrozen_G

In [ ]:
# Multi-Attribute Node Labeling for GNN Fairness Auditing
def get_nodes_labels_real():
    # 1. Building the graph structure from edges.txt
    graph_df = pd.read_csv('edges.txt', sep="\t", header=None)
    graph_df.columns = ['s', 't']
    
    edges = [(row.s, row.t) for _, row in graph_df.iterrows()]
    input_G = nx.from_edgelist(edges)
    unfrozen_G = nx.Graph(input_G)
    nodes_G = list(input_G.nodes())

    # 2. Fetching and Categorizing Attributes Generically
    data = pd.read_csv('attr.txt', sep="\t", header=None)
    
    extra_nodes = []
    attr1_labels, attr2_labels = {}, {}

    for _, row in data.iterrows():
        node_id = row.iloc[0] # Assumes ID is in the first column
        if node_id not in nodes_G:
            continue
    
        val_attr1 = row.iloc[2]
        if val_attr1 < 20:
            attr1_labels[node_id] = 0
        elif val_attr1 == 20:
            attr1_labels[node_id] = 1
        else:
            attr1_labels[node_id] = 2
            extra_nodes.append(node_id)

        val_attr2 = row.iloc[1]
        attr2_labels[node_id] = 0 if val_attr2 < 7 else 1
    
    # 3. Pruning the Graph based on Attribute 1 constraints
    for node in input_G.nodes():
        if node in extra_nodes:
            unfrozen_G.remove_node(node)
    
    # 4. Creating finalized index-aligned label arrays for GNN training 
    labels_attr1 = np.array([attr1_labels[node] for node in unfrozen_G.nodes()])
    labels_attr2 = np.array([attr2_labels[node] for node in unfrozen_G.nodes()])
    
    nodes = np.arange(len(unfrozen_G.nodes()))
    
    # Identify indices for Adversarial Balancing 
    attr1_zero, attr1_one = nodes[labels_attr1 == 0], nodes[labels_attr1 == 1]
    attr2_zero, attr2_one = nodes[labels_attr2 == 0], nodes[labels_attr2 == 1]

    return attr1_zero, attr1_one, labels_attr1, attr2_zero, attr2_one, labels_attr2

In [ ]:
# Balanced Topological Oversampling for Fair GNN Training
def get_idxs(n, nodes_zero, nodes_one, idxs_size):
    # n: Total number of nodes in the graph
    # nodes_zero/one: Lists of indices belonging to specific demographic groups
    
    idxs_zeros = nodes_zero[:]
    idxs_ones = nodes_one[:]
    
    # Balancing Group 0 
    rep_policy = False
    if len(nodes_zero) < idxs_size:
        if idxs_size > 2 * len(nodes_zero):
            rep_policy = True
        zero_draws = np.random.choice(nodes_zero, size=idxs_size - len(nodes_zero), replace=rep_policy)
        idxs_zeros = np.concatenate((idxs_zeros, zero_draws))
    
    # Balancing Group 1 
    rep_policy = False
    if len(nodes_one) < idxs_size:
        if idxs_size > 2 * len(nodes_one):
            rep_policy = True
        one_draws = np.random.choice(nodes_one, size=idxs_size - len(nodes_one), replace=rep_policy)
        idxs_ones = np.concatenate((idxs_ones, one_draws))

    # Asserting that both groups have equal representation for the Adversarial Game
    assert len(idxs_zeros) == len(idxs_ones)

    return np.arange(n), idxs_zeros, idxs_ones

In [ ]:
# Topological Homophily and Community Connectivity Audit
def print_edges(G, nodes_zero, nodes_one):
    # Analyzing the structural links that guide GNN Message Passing
    zero_edges = 0
    one_edges = 0
    accross_edges = 0

    # Iterating through the edges defined in your edges.txt dataset
    for (v1, v2) in G.edges():
        # Check if the connection is internal to Group 0 
        if v1 in nodes_zero and v2 in nodes_zero:
            zero_edges += 1
        # Check if the connection is internal to Group 1 
        elif v1 in nodes_one and v2 in nodes_one:
            one_edges += 1
        # Check if the connection bridges the two demographics
        elif (v1 in nodes_one and v2 in nodes_zero) or (v1 in nodes_zero and v2 in nodes_one):
            accross_edges += 1

    print(f"Edges within zero community: {zero_edges}")
    print(f"Edges within one community: {one_edges}")
    print(f"Edges across communities: {accross_edges}")

RUNNING THE EXPERIMENTS

In [ ]:
import time

# Configuration for GNN Architecture
embedding_size = 30
first_order_imp = 'no_f1' # Set to 'with_f1' to enable Laplacian Regularization
alpha = 0.05

# 1. Building the GNN-Ready Graph and Adjacency Matrix (A)
# X here acts as the Node Feature Matrix for GCN Message Passing
G, A, input_G = get_graph_real()
n = len(G.nodes())
feature_matrix = tf.cast(A, tf.float32) # Using structure as features for this GNN variant

# 2. Extracting Demographic Clusters for Multi-Attribute Fairness
attr1_zero, attr1_one, labels_attr1, attr2_zero, attr2_one, labels_attr2 = get_nodes_labels_real()
idxs_size = np.max([len(attr1_zero), len(attr1_one), len(attr2_zero), len(attr2_one)])

print_edges(G, attr1_zero, attr1_one)

# 3. Generating Balanced Indices for Adversarial Training
idxs, idxs1_zeros, idxs1_ones = get_idxs(n, attr1_zero, attr1_one, idxs_size)
_, idxs2_zeros, idxs2_ones = get_idxs(n, attr2_zero, attr2_one, idxs_size)

# 4. Building GCN-based Encoder and MLP Decoder
encoder = build_encoder(embedding_size)
decoder = build_decoder(embedding_size, n)
auto_encoder = build_ae(encoder, decoder, n)

# 5. Initializing Dual Discriminators 
discriminator1 = build_discriminator(embedding_size)
discriminator2 = build_discriminator(embedding_size)

# 6. Optimized Pre-training Phase
pre_optimizer_embd = tf.keras.optimizers.Adam()
pre_optimizer_disc1 = tf.keras.optimizers.Adam()
pre_optimizer_disc2 = tf.keras.optimizers.Adam()

print("Starting GNN Pre-training...")
time1 = time.time()

# Pre-train GCN to capture neighborhood topology
pretrain_embd(A, feature_matrix, idxs, encoder, decoder, auto_encoder, pre_optimizer_embd, first_order_imp, alpha)

# Pre-train Discriminators to identify initial bias in neighborhood aggregations
pretrain_disc(A, feature_matrix, idxs1_zeros, idxs1_ones, encoder, discriminator1, pre_optimizer_disc1)
pretrain_disc(A, feature_matrix, idxs2_zeros, idxs2_ones, encoder, discriminator2, pre_optimizer_disc2)

# Generate baseline "Graph-Aware" embeddings before adversarial debiasing
pre_embds = encoder([feature_matrix, A])
print(f'Pre-training completed in {time.time() - time1:.2f} seconds.')

In [ ]:
# Multi-Attribute Adversarial Training Execution
# Initializing optimizers for the GNN and its two adversaries
ae_optimizer = tf.keras.optimizers.Adam()
disc1_optimizer = tf.keras.optimizers.Adam()
disc2_optimizer = tf.keras.optimizers.Adam()

print("Launching Adversarial Game: Balancing Topology vs. Fairness...")

# Execution of the Joint Training Loop
adversarial_train(A, feature_matrix, 
                  idxs1_zeros, idxs1_ones, 
                  idxs2_zeros, idxs2_ones, 
                  encoder, decoder, auto_encoder, 
                  discriminator1, discriminator2, 
                  ae_optimizer, disc1_optimizer, disc2_optimizer)

# Generating the Final 'Fair' Embeddings
fair_embds = encoder([feature_matrix, A])

print('Adversarial training completed successfully.')
time_spent = np.round(time.time() - time1, 2)

print(f"Total Training Time: {time_spent} seconds")

EXPERIMENT SETTINGS

In [ ]:
# Experimental Setup for Fair Influence Maximization
# Number of clusters (communities) to identify in the latent space
N_CLUSs = [4]

# Range of seeds to pick PER cluster (Total seeds = N_CLUS * n_seeds)
n_seedss = [1, 2, 3, 4, 5, 6, 7, 8, 10]

# Selection Strategy: 're-cluster' ensures that the GNN-identified 
# communities maintain demographic diversity among the influencers
strategy = 're-cluster'

In [ ]:
import pickle

# Loading the Greedy Benchmark for Performance Comparison

with open('saved_models/greedy_seeds.pickle', 'rb') as f:
    # Loading the pre-calculated greedy seeds to compare against our GNN approach
    greedy_seeds_raw = pickle.load(f)

# Ensuring the greedy seeds are formatted correctly for the IC simulation
greedy_seeds = np.array(greedy_seeds_raw[0])

print(f"Loaded {len(greedy_seeds)} Greedy Seeds for benchmarking.")

In [ ]:
# Final Performance & Fairness Evaluation: GNN vs. Greedy Baseline
results_log = []
seeds_cur = 0

print(f"{'Seeds':<8} | {'Total Reach':<12} | {'Fairness (Group 0/1)':<20}")
print("-" * 50)

for N_CLUS in N_CLUSs:
    for n_seeds in n_seedss:
        # 1. Select seeds using your Fair GNN embeddings
        # The GNN has already scrubbed bias from the neighborhood aggregation
        fair_seeds = get_seeds(N_CLUS, fair_embds, idxs, labels_attr1, attr1_zero, attr1_one, strategy, n_seeds)

        # 2. Simulate influence spread (2000 trials for statistical robustness)
        # Testing for both Attribute 1 and Attribute 2
        total_fair, fair_frac, z_f1, o_f1, z_f2, o_f2 = repeated_IC(G, fair_seeds, 'fair', 2000, 0.01)

        # 3. Simulate influence spread for the Greedy baseline
        # We use the pre-calculated greedy_seeds loaded earlier
        total_greedy, greedy_frac, z_g1, o_g1, z_g2, o_g2 = repeated_IC(
            G, np.reshape(greedy_seeds[seeds_cur], newshape=(-1,)), 'greedy', 2000, 0.01
        )

        # 4. Display comparative results
        # Observe the gap between z_f1 and o_f1 compared to z_g1 and o_g1
        row_fair = [n_seeds, total_fair, fair_frac, z_f1, o_f1, z_f2, o_f2]
        row_greedy = [n_seeds, total_greedy, greedy_frac, z_g1, o_g1, z_g2, o_g2]
        
        print(f"Fair GNN (Seeds/Clus={n_seeds}): {total_fair:<12} | {z_f1}/{o_f1}")
        print(f"Greedy   (Seeds/Clus={n_seeds}): {total_greedy:<12} | {z_g1}/{o_g1}")
        print("-" * 50)
        
        results_log.append({'fair': row_fair, 'greedy': row_greedy})
        seeds_cur += 1

In [ ]:
import pickle

with open('gnn_fair_results.pickle', 'wb') as f:
    # Saving the Adjacency Matrix and both sets of embeddings
    # fair_embds: The final output of our GCN-Adversarial pipeline
    # pre_embds: The baseline GCN output without fairness constraints
    pickle.dump([
        G, 
        embedding_size, 
        fair_embds, 
        pre_embds, 
        idxs, 
        labels_attr1, 
        attr1_zero, 
        attr1_one
    ], f)

print('Fair GNN Results Saved Successfully.')

In [ ]:
import pickle

# Persisting Demographic Labels & Graph Topology

with open('labels_nodes.pickle', 'wb') as f:
    # G: The NetworkX graph built from edges.txt
    # idxs: The node indices used for training alignment
    # attr1_zero/one: Demographic splits for Attribute 1
    # attr2_zero/one: Demographic splits for Attribute 2
    pickle.dump([
        G, 
        idxs, 
        labels_attr1, attr1_zero, attr1_one, 
        labels_attr2, attr2_zero, attr2_one
    ], f)

print('Structural and demographic labels saved successfully.')

In [ ]:
import scipy as sc
import pickle

# Restoring Fair GNN State for Influence Evaluation
# Loading the saved graph structure and the fair/baseline embeddings
with open('saved_models/embds_2_1.pickle', 'rb') as f:
    # We unpack the specific objects needed to run the 'repeated_IC' audit
    [G, embedding_size, fair_embds, pre_embds, idxs, labels_attr1, attr1_zero, attr1_one] = pickle.load(f)

print(f"Successfully restored GNN state for {len(G.nodes())} nodes.")
print(f"Fair Embedding Shape: {fair_embds.shape}")

In [ ]:
# Statistical Significance Testing: Fair GNN vs. Pre-trained Baseline
p_values = []
N_CLUSs = [4]
n_seedss = [1, 2, 3, 4, 5, 6, 7, 8, 10]
strategy = 're-cluster'

for N_CLUS in N_CLUSs:
    for n_seeds in n_seedss:
        print(f"Analyzing significance for n_seeds={n_seeds}...")
        
        # 1. Generate Fair Seeds and simulate Influence Reach
        fair_seeds = get_seeds(N_CLUS, fair_embds, idxs, labels_attr1, attr1_zero, attr1_one, strategy, n_seeds)
        # Note: Using 2000 trials to ensure a normal distribution for the T-Test
        fair_records = get_IC_influenced(G, fair_seeds, 2000, 0.01)
        
        # 2. Generate Pre-trained (Biased) Seeds and simulate Influence Reach
        pre_seeds = get_seeds(N_CLUS, pre_embds, idxs, labels_attr1, attr1_zero, attr1_one, strategy, n_seeds)
        pre_records = get_IC_influenced(G, pre_seeds, 2000, 0.01) # Keep imp_prob consistent at 0.01
        
        # 3. Perform Independent T-Test on the 'Total Reach' arrays
        # We compare fair_records[0] (total count) vs pre_records[0]
        t_stat, p_value = sc.stats.ttest_ind(fair_records[0], pre_records[0])
        p_values.append(p_value)

# High p-values (> 0.05) indicate that your Fair GNN achieves similar reach to the baseline
# while being significantly more fair demographically.
print("P-Values across seed counts:", p_values)